# Notebook goal

```text
Silver responsibilities:

- correct data types
- standardize values
- validate business rules
- remove invalid records
- remove duplicate records
- preserve useful ingestion metadata

Input:

ride_lakehouse_dev.bronze.drivers
ride_lakehouse_dev.bronze.rides
ride_lakehouse_dev.bronze.payments

Output:

ride_lakehouse_dev.silver.drivers
ride_lakehouse_dev.silver.rides
ride_lakehouse_dev.silver.payments
```

# Streaming is still valid in sliver transformation
Silver layer is source of truth so it needs to clean data on fly (while data is coming). In this case we need to use streaming.

# Do we need auto loader
No, because it is usefull when non-tabular files coming into our cloud storage. Now, we have databricks table that we can keep track straming data. 

# Checkpoint
We need check point for silver layer to because straming is still vail and we need to knwo which data is alread cleaned or not

```text
checkpoints/
│
├── bronze_drivers/
├── bronze_rides/
├── bronze_payments/
│
├── silver_drivers/      ← NEW
├── silver_rides/        ← NEW
└── silver_payments/     ← NEW

 bronze → silver
```

In [0]:
BASE_PATH = "/Volumes/ride_lakehouse_dev/bronze/streaming_files"

SILVER_DRIVERS_CHECKPOINT = (
    f"{BASE_PATH}/checkpoints/silver_drivers"
)

SILVER_RIDES_CHECKPOINT = (
    f"{BASE_PATH}/checkpoints/silver_rides"
)

SILVER_PAYMENTS_CHECKPOINT = (
    f"{BASE_PATH}/checkpoints/silver_payments"
)

We donWt need volume because silver layer is structured data already. Volume is for non-tabular data

In [0]:
from pyspark.sql import functions as F

# Reading drivers as a stream
This provides us a continous data source for writing/cleaning

In [0]:
bronze_drivers_stream = (
    spark.readStream
        .table("ride_lakehouse_dev.bronze.drivers")
)

In [0]:
#bronze_drivers_stream.isStreaming

True

# Correct the data types
```text
generated_at
    ↓
string
    ↓
timestamp
```

In [0]:
# silver_drivers_stream = (
#     bronze_drivers_stream

#         .withColumn(
#             "generated_at",
#             F.to_timestamp("generated_at")
#         )
# )

# Standardize strings

In [0]:
# silver_drivers_stream = (
#     bronze_drivers_stream


#         .withColumn(
#             "name",
#             F.trim(F.col("name"))
#         )

#         .withColumn(
#             "car_number",
#             F.upper(
#                 F.trim(F.col("car_number"))
#             )
#         )
# )

# Validate driver IDs- rating-experience- required text columns

In [0]:
# silver_drivers_stream = (
#     bronze_drivers_stream
#         .filter(
#             F.col("id").isNotNull()
#             &
#             (F.trim(F.col("id")) != "")
#         )

#         .filter(
#             F.col("rating").between(0.0, 5.0)
#         )

#         .filter(
#             F.col("experience").between(0, 60)
#         )

#         .filter(
#             F.col("name").isNotNull()
#             &
#             F.col("car_number").isNotNull()
#         )
# )

# Deduplicate
Foe this project spark can detect duplicate values in abtch but if data keeps continue to flow our silver layer there would be some problem.

1- watermark is one of solution it can remember state for certain time (like 2 hours) but after that time it will forget and let duplicated value go

2- In massive project developers use RocksDB, a highly optimized database that stores this state on the server's much larger solid-state drives (SSDs) rather than relying solely on RAM.

In [0]:
# silver_drivers_stream = (
#     bronze_drivers_stream
#         .dropDuplicates(["id"])
# )

# Whole code for drivers silver table
Lastly we add metadata for silver table _silver_processed_at

In [0]:
silver_drivers_stream = (
    bronze_drivers_stream

        .withColumn(
            "generated_at",
            F.to_timestamp("generated_at")
        )

        .withColumn(
            "name",
            F.trim(F.col("name"))
        )

        .withColumn(
            "car_number",
            F.upper(
                F.trim(F.col("car_number"))
            )
        )

        .filter(
            F.col("id").isNotNull()
            &
            (F.trim(F.col("id")) != "")
        )

        .filter(
            F.col("name").isNotNull()
        )

        .filter(
            F.col("car_number").isNotNull()
        )

        .filter(
            F.col("rating").between(0.0, 5.0)
        )

        .filter(
            F.col("experience").between(0, 60)
        )

        .dropDuplicates(["id"])

        .withColumn(
            "_silver_processed_at",
            F.current_timestamp()
        )
)

# Write silver table

In [0]:
silver_drivers_query = (
    silver_drivers_stream
        .writeStream

        .format("delta")

        .outputMode("append")

        .option(
            "checkpointLocation",
            SILVER_DRIVERS_CHECKPOINT
        )

        .option(
            "mergeSchema",
            "true"
        )

        .trigger(
            availableNow=True
        )

        .toTable(
            "ride_lakehouse_dev.silver.drivers"
        )
)

silver_drivers_query.awaitTermination()

# test

In [0]:
# display(
#     spark.table(
#         "ride_lakehouse_dev.silver.drivers"
#     ).limit(20)
# )

car_number,experience,generated_at,id,name,rating,_rescued_data,_ingested_at,_source_file,_silver_processed_at
GZ-4394,17,2026-09-29T11:49:09.009Z,D-20260929114909008713-000,Roberto Smith,1.46,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114909008713.json,2026-09-29T18:29:27.987Z
QU-4530,1,2026-09-29T11:49:13.448Z,D-20260929114913448298-001,Mr. Keith Yoder,1.34,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114913448298.json,2026-09-29T18:29:27.987Z
CP-2391,18,2026-09-29T11:49:16.838Z,D-20260929114916836637-009,Mr. Logan Atkins,4.26,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114916836637.json,2026-09-29T18:29:27.987Z
KH-0372,13,2026-09-29T11:49:16.837Z,D-20260929114916836637-003,Thomas Green,3.77,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114916836637.json,2026-09-29T18:29:27.987Z
HV-2885,6,2026-09-29T11:49:13.448Z,D-20260929114913448298-000,Daniel Soto,2.0,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114913448298.json,2026-09-29T18:29:27.987Z
ZI-5968,16,2026-09-29T11:49:13.449Z,D-20260929114913448298-007,Maria Hernandez,3.16,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114913448298.json,2026-09-29T18:29:27.987Z
OM-4253,8,2026-09-29T11:49:09.010Z,D-20260929114909008713-006,Kevin Phillips,3.32,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114909008713.json,2026-09-29T18:29:27.987Z
IR-6374,3,2026-09-29T11:49:13.449Z,D-20260929114913448298-002,Courtney Douglas,1.95,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114913448298.json,2026-09-29T18:29:27.987Z
ZI-0210,6,2026-09-29T11:45:44.668Z,D-20260929114544667600-005,Scott Conley,1.88,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114544667600.json,2026-09-29T18:29:27.987Z
VW-7459,3,2026-09-29T11:49:09.009Z,D-20260929114909008713-003,Michael Gray,4.84,null,2026-09-29T13:01:26.940Z,/Volumes/ride_lakehouse_dev/bronze/streaming_files/landing/drivers/drivers_batch_20260929114909008713.json,2026-09-29T18:29:27.987Z


In [0]:
# spark.table(
#     "ride_lakehouse_dev.silver.drivers"
# ).printSchema()

root
 |-- car_number: string (nullable = true)
 |-- experience: long (nullable = true)
 |-- generated_at: timestamp (nullable = true)
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = true)



In [0]:
# bronze_driver_count = (
#     spark.table(
#         "ride_lakehouse_dev.bronze.drivers"
#     ).count()
# )

# silver_driver_count = (
#     spark.table(
#         "ride_lakehouse_dev.silver.drivers"
#     ).count()
# )

# print(
#     f"Bronze drivers: {bronze_driver_count}"
# )

# print(
#     f"Silver drivers: {silver_driver_count}"
# )

Bronze drivers: 40
Silver drivers: 40


# Transoform Rides

In [0]:
bronze_rides_stream = (
    spark.readStream
        .table(
            "ride_lakehouse_dev.bronze.rides"
        )
)

In [0]:
silver_rides_stream = (
    bronze_rides_stream

        
        .withColumn(
            "ride_timestamp",
            F.to_timestamp("ride_timestamp")
        )

        .withColumn(
            "generated_at",
            F.to_timestamp("generated_at")
        )

        
        .withColumn(
            "pickup_city",
            F.trim(F.col("pickup_city"))
        )

        .withColumn(
            "dropoff_city",
            F.trim(F.col("dropoff_city"))
        )

        
        .filter(
            F.col("ride_id").isNotNull()
        )

        .filter(
            F.col("driver_id").isNotNull()
        )

        .filter(
            F.col("customer_id").isNotNull()
        )

        
        .filter(
            F.col("pickup_city").isNotNull()
        )

        .filter(
            F.col("dropoff_city").isNotNull()
        )

        
        .filter(
            F.col("distance_km") > 0
        )

        .filter(
            F.col("fare_amount") >= 0
        )

        
        .dropDuplicates(["ride_id"])

        
        .withColumn(
            "_silver_processed_at",
            F.current_timestamp()
        )
)

# Write rides table

In [0]:
silver_rides_query = (
    silver_rides_stream
        .writeStream

        .format("delta")

        .outputMode("append")

        .option(
            "checkpointLocation",
            SILVER_RIDES_CHECKPOINT
        )

        .option(
            "mergeSchema",
            "true"
        )

        .trigger(
            availableNow=True
        )

        .toTable(
            "ride_lakehouse_dev.silver.rides"
        )
)

silver_rides_query.awaitTermination()

# Transform payments

In [0]:
bronze_payments_stream = (
    spark.readStream
        .table(
            "ride_lakehouse_dev.bronze.payments"
        )
)

In [0]:
silver_payments_stream = (
    bronze_payments_stream

        # Correct timestamp types
        .withColumn(
            "payment_timestamp",
            F.to_timestamp("payment_timestamp")
        )

        .withColumn(
            "generated_at",
            F.to_timestamp("generated_at")
        )

        # Standardize payment method
        .withColumn(
            "payment_method",
            F.lower(
                F.trim(
                    F.col("payment_method")
                )
            )
        )

        # Required identifiers
        .filter(
            F.col("payment_id").isNotNull()
        )

        .filter(
            F.col("ride_id").isNotNull()
        )

        # Business validation
        .filter(
            F.col("amount") >= 0
        )

        .filter(
            F.col("payment_method").isin(
                "card",
                "cash",
                "wallet"
            )
        )

        # Remove duplicates
        .dropDuplicates(["payment_id"])

        # Silver processing metadata
        .withColumn(
            "_silver_processed_at",
            F.current_timestamp()
        )
)

# Write payment table

In [0]:
silver_payments_query = (
    silver_payments_stream
        .writeStream

        .format("delta")

        .outputMode("append")

        .option(
            "checkpointLocation",
            SILVER_PAYMENTS_CHECKPOINT
        )

        .option(
            "mergeSchema",
            "true"
        )

        .trigger(
            availableNow=True
        )

        .toTable(
            "ride_lakehouse_dev.silver.payments"
        )
)

silver_payments_query.awaitTermination()

# test

In [0]:
%sql
-- SELECT
--     r.ride_id,
--     r.driver_id

-- FROM ride_lakehouse_dev.silver.rides r

-- LEFT ANTI JOIN
--     ride_lakehouse_dev.silver.drivers d

-- ON r.driver_id = d.id;

ride_id,driver_id
